# Section 04 lab — Aggregations and Spark SQL

The first two exercises use Python rows so the aggregation is easy to verify. The final exercise applies the same ideas to the course data.


## Exercise 1 — Aggregate with the DataFrame API


In [ ]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .master('local[2]')
         .appName('PySpark-Course')
         .config('spark.sql.shuffle.partitions', '4')
         .config('spark.sql.adaptive.enabled', 'true')
         .config('spark.sql.session.timeZone', 'UTC')
         .getOrCreate())
from pyspark.sql import functions as F

sales_rows = [('O1', 'Accessories', 2, 500.0), ('O2', 'Audio', 1, 1200.0),
              ('O3', 'Accessories', 3, 750.0), ('O4', 'Audio', 2, 2400.0)]
sales = spark.createDataFrame(sales_rows, ['order_id', 'category', 'units', 'revenue'])
summary = (sales.groupBy('category')
           .agg(F.countDistinct('order_id').alias('orders'),
                F.sum('units').alias('units'),
                F.sum('revenue').alias('revenue'))
           .orderBy('category'))
summary.show()
assert summary.agg(F.sum('revenue')).first()[0] == 4850.0


## Exercise 2 — Produce the same result with Spark SQL


In [ ]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .master('local[2]')
         .appName('PySpark-Course')
         .config('spark.sql.shuffle.partitions', '4')
         .config('spark.sql.adaptive.enabled', 'true')
         .config('spark.sql.session.timeZone', 'UTC')
         .getOrCreate())

sales_rows = [('O1', 'Accessories', 2, 500.0), ('O2', 'Audio', 1, 1200.0),
              ('O3', 'Accessories', 3, 750.0), ('O4', 'Audio', 2, 2400.0)]
sales = spark.createDataFrame(sales_rows, ['order_id', 'category', 'units', 'revenue'])
sales.createOrReplaceTempView('sales')
result = spark.sql("""
SELECT category, COUNT(DISTINCT order_id) AS orders,
       SUM(units) AS units, SUM(revenue) AS revenue
FROM sales GROUP BY category ORDER BY category
""")
result.show()
assert result.agg({'revenue': 'sum'}).first()[0] == 4850.0


## Exercise 3 — Reconcile completed revenue from the course data


In [ ]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .master('local[2]')
         .appName('PySpark-Course')
         .config('spark.sql.shuffle.partitions', '4')
         .config('spark.sql.adaptive.enabled', 'true')
         .config('spark.sql.session.timeZone', 'UTC')
         .getOrCreate())
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

transaction_schema = StructType([
    StructField('transaction_id', StringType()), StructField('transaction_date', StringType()),
    StructField('customer_id', StringType()), StructField('product_id', StringType()),
    StructField('quantity', StringType()), StructField('unit_price', StringType()),
    StructField('status', StringType())])
product_schema = StructType([
    StructField('product_id', StringType()), StructField('product_name', StringType()),
    StructField('category', StringType()), StructField('list_price', DoubleType())])
customer_schema = StructType([
    StructField('customer_id', StringType()), StructField('customer_name', StringType()),
    StructField('city', StringType())])
raw = spark.read.schema(transaction_schema).option('header', True).csv('../data/transactions.csv')
products = spark.read.schema(product_schema).json('../data/products.json')
customers = spark.read.schema(customer_schema).option('header', True).csv('../data/customers.csv')
typed = (raw.dropDuplicates(['transaction_id'])
         .withColumn('order_date', F.expr('try_cast(transaction_date as date)'))
         .withColumn('qty', F.expr('try_cast(quantity as int)'))
         .withColumn('price', F.expr('try_cast(unit_price as decimal(12,2))')))
clean = typed.filter(F.col('order_date').isNotNull() & F.col('qty').isNotNull() & (F.col('qty') > 0) & F.col('price').isNotNull() & (F.col('price') >= 0) & (F.length(F.trim('customer_id')) > 0) & (F.length(F.trim('product_id')) > 0)).withColumn('line_total', F.col('qty') * F.col('price'))
sales = (clean.filter(F.col('status') == 'COMPLETED')
         .join(products, 'product_id', 'inner')
         .join(customers, 'customer_id', 'inner'))
summary = sales.groupBy('category').agg(F.countDistinct('transaction_id').alias('orders'), F.sum('line_total').alias('revenue')).orderBy('category')
summary.show()
total = summary.agg(F.sum('revenue')).first()[0]
print('Total revenue:', total)
assert str(total) == '342228.00'
